# Entrenamiento del detector de matrículas YOLO11n

Entrena y evalúa YOLO11n sobre `datos/dataset_limpio`.
Los datos se descargan antes con `descargar_datos.ipynb`. El notebook asume que se ejecuta desde la carpeta `P4`.

In [ ]:
from pathlib import Path

import torch
from ultralytics import YOLO

AQUI = Path.cwd()  # carpeta P4
DATASET = AQUI.parent / "datos" / "dataset_limpio"
RUNS = AQUI / "runs"
MODELO = RUNS / "matriculas_yolo11n" / "weights" / "best.pt"

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # GPU si hay, si no CPU
print("Dispositivo:", DEVICE)

SOLO_TEST = False  # True = no entrenar, solo evaluar best.pt en test

In [ ]:
if not SOLO_TEST:
    # Con AMP la pérdida se iba a nan en la época 5, por eso amp=False
    YOLO(str(AQUI / "yolo11n.pt")).train(data=str(DATASET / "data.yaml"), epochs=60, imgsz=640, batch=8,
                                         device=DEVICE, workers=2, amp=False,
                                         project=str(RUNS), name="matriculas_yolo11n", exist_ok=True)

In [ ]:
m = YOLO(str(MODELO)).val(data=str(DATASET / "data.yaml"), split="test", project=str(RUNS),
                          name="matriculas_yolo11n_test", exist_ok=True, verbose=False)
print(f"Test | Precisión {m.box.mp:.3f} | Recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")